In [ ]:
import pandas as pd
import numpy as np
import os

#  LOADING UP THE DATASETS

base_path = None

for dirname, _, filenames in os.walk('/kaggle/input'):
    if 'train.csv' in filenames:
        base_path = dirname
        break

if base_path is None:
    raise FileNotFoundError("Could not find train.csv inside /kaggle/input")

print("Dataset found at:", base_path)

train_df = pd.read_csv(os.path.join(base_path, 'train.csv'))
valid_df = pd.read_csv(os.path.join(base_path, 'valid.csv'))
test_df  = pd.read_csv(os.path.join(base_path, 'test.csv'))

train_A = train_df[train_df['part'] == 'A'].copy()
train_B = train_df[train_df['part'] == 'B'].copy()
train_C = train_df[train_df['part'] == 'C'].copy()
train_D = train_df[train_df['part'] == 'D'].copy()

valid_A = valid_df[valid_df['part'] == 'A'].copy()
valid_B = valid_df[valid_df['part'] == 'B'].copy()
valid_C = valid_df[valid_df['part'] == 'C'].copy()
valid_D = valid_df[valid_df['part'] == 'D'].copy()

test_A = test_df[test_df['part'] == 'A'].copy()
test_B = test_df[test_df['part'] == 'B'].copy()
test_C = test_df[test_df['part'] == 'C'].copy()
test_D = test_df[test_df['part'] == 'D'].copy()


# HELPER FUNCTIONS


categorical_cols_AB = ['dow', 'month_phase', 'dept', 'sensor_class', 'building_type']
categorical_cols_CD = ['dow', 'month_phase', 'dept', 'sensor_class', 'building_type',
                       'entity_id', 'holiday_flag']

def extract_unique(df, categorical_cols):
    colwise_unique = []
    for col in categorical_cols:
        unique = set()
        for i in df[col].dropna():
            unique.add(i)
        unique = list(unique)
        unique.sort()
        colwise_unique.append(unique)
    return colwise_unique

def one_hot(df, categorical_cols, colwise_unique):
    df = df.copy()
    new_cols = {}
    for i in range(len(categorical_cols)):
        col = categorical_cols[i]
        for x in colwise_unique[i]:
            new_col = col + "_" + str(x)
            new_cols[new_col] = (df[col] == x).astype(int)
        df.drop(columns=[col], inplace=True)
    if new_cols:
        df = pd.concat([df, pd.DataFrame(new_cols, index=df.index)], axis=1)
    return df

def median_imputation(df, trainmedians=None):
    df = df.copy()
    if trainmedians is None:
        trainmedians = {}
        for x in ["m1","m2","m3","m4","m5","m6"]:
            trainmedians[x] = df[x].median()
    for x in ["m1","m2","m3","m4","m5","m6"]:
        df[x] = df[x].fillna(trainmedians[x])
    return df, trainmedians

def normalize(df, trainstats=None):
    df = df.copy()
    norm_cols = [col for col in df.columns
                 if col[0] == 'x' or col in ('m1', 'm2', 'm3', 'm4', 'm5', 'm6',
                                              'z1', 'z2', 'z3', 'z4', 'z5', 'z6',
                                              'z7', 'z8', 'timestamp', 'y_lag1', 'y_lag7')]
    if trainstats is None:
        trainstats = {}
        for col in norm_cols:
            trainstats[col] = (df[col].mean(), df[col].std())
    for col in norm_cols:
        if col not in trainstats:
            continue
        mu, sigma = trainstats[col]
        if pd.isna(mu) or pd.isna(sigma) or sigma < 1e-8:
            df[col] = 0.0
        else:
            df[col] = (df[col] - mu) / sigma
            df[col] = df[col].fillna(0.0)
    return df, trainstats

def robust_normalize(df, stats=None, clip_threshold=3.5):
    df = df.copy()
    norm_cols = [col for col in df.columns
                 if col[0] == 'x' or col in ('m1','m2','m3','m4','m5','m6',
                                             'z1','z2','z3','z4','z5','z6',
                                             'z7','z8','timestamp','y_lag1','y_lag7')]
    MAD_SCALE = 1.4826
    if stats is None:
        stats = {}
        for col in norm_cols:
            col_data = df[col].values
            median = np.nanmedian(col_data)
            mad = np.nanmedian(np.abs(col_data - median))
            if np.isnan(mad) or mad < 1e-8:
                mad = 1.0
            stats[col] = {'median': median, 'mad': mad}
    for col in norm_cols:
        if col not in stats:
            continue
        median = stats[col]['median']
        mad = stats[col]['mad']
        scaled_mad = mad * MAD_SCALE
        df[col] = df[col].fillna(median)
        lower = median - clip_threshold * scaled_mad
        upper = median + clip_threshold * scaled_mad
        df[col] = np.clip(df[col], lower, upper)
        if scaled_mad < 1e-8:
            df[col] = 0.0
        else:
            df[col] = (df[col] - median) / scaled_mad
    return df, stats

def clip_outliers(df, clip_stats=None):
    df = df.copy()
    x_cols = [col for col in df.columns if col.startswith('x')]
    if clip_stats is None:
        clip_stats = {}
        for col in x_cols:
            mu    = df[col].mean()
            sigma = df[col].std()
            clip_stats[col] = (mu - 3 * sigma, mu + 3 * sigma)
    for col in x_cols:
        if col in df.columns:
            lo, hi  = clip_stats[col]
            df[col] = df[col].clip(lower=lo, upper=hi)
    return df, clip_stats

def normalize_robust(df, stats=None):
    df = df.copy()
    x_cols     = [col for col in df.columns if col.startswith('x')]
    other_cols = [col for col in df.columns
                  if col in ('m1', 'm2', 'm3', 'm4', 'm5', 'm6',
                             'timestamp', 'y_lag1', 'y_lag7', 'entity_mean_y')]
    if stats is None:
        stats = {}
        for col in x_cols:
            median     = df[col].median()
            mad        = (df[col] - median).abs().median()
            stats[col] = ('robust', median, mad)
        for col in other_cols:
            mu         = df[col].mean()
            sigma      = df[col].std()
            stats[col] = ('standard', mu, sigma)
    for col, info in stats.items():
        if col not in df.columns:
            continue
        if info[0] == 'robust':
            _, median, mad = info
            scale = 1.4826 * mad
            if pd.isna(median) or pd.isna(mad) or scale < 1e-8:
                df[col] = 0.0
            else:
                df[col] = (df[col] - median) / scale
                df[col] = df[col].fillna(0.0)
        elif info[0] == 'standard':
            _, mu, sigma = info
            if pd.isna(mu) or pd.isna(sigma) or sigma < 1e-8:
                df[col] = 0.0
            else:
                df[col] = (df[col] - mu) / sigma
                df[col] = df[col].fillna(0.0)
    return df, stats

def createX_bias(df, actual_cols):
    X    = df[actual_cols].values.astype(np.float64)
    X    = np.nan_to_num(X, nan=0.0)
    bias = np.ones((X.shape[0], 1), dtype=np.float64)
    return np.hstack([bias, X])

def createX_no_bias(df, actual_cols):
    X = df[actual_cols].values.astype(np.float64)
    return np.nan_to_num(X, nan=0.0)

def createy(df):
    return df['y'].values.astype(np.float64)

def predict(w, X):
    return X @ w

def mae(y_true, y_pred):
    return np.mean(np.abs(y_true - y_pred))

def ridge_closed_form(X, y, lamda=1.0):
    n, d = X.shape
    ridge = lamda * np.eye(d)
    ridge[0, 0] = 0.0
    A = X.T @ X + ridge
    b = X.T @ y
    return np.linalg.solve(A, b)

def add_bias(X):
    return np.hstack([np.ones((X.shape[0], 1), dtype=np.float64), X])

def build_custom_poly_features(X):
    X = np.asarray(X, dtype=np.float64)
    n, d = X.shape
    feats = []
    # 1) Linear
    feats.append(X)
    # 2) Quadratic interactions (i <= j)
    quad_terms = []
    for i in range(d):
        xi = X[:, i:i+1]
        for j in range(i, d):
            quad_terms.append(xi * X[:, j:j+1])
    if quad_terms:
        feats.append(np.hstack(quad_terms))
    # 3) Higher powers (no interactions)
    feats.append(X ** 3)
    feats.append(X ** 4)
    feats.append(X ** 5)
    feats.append(X ** 6)
    return np.hstack(feats).astype(np.float64)

# PART A — Ridge Regression

unique_A = extract_unique(train_A, categorical_cols_AB)
train_A = one_hot(train_A, categorical_cols_AB, unique_A)
valid_A = one_hot(valid_A, categorical_cols_AB, unique_A)
test_A  = one_hot(test_A,  categorical_cols_AB, unique_A)

train_A, train_medians_A = median_imputation(train_A)
valid_A, _ = median_imputation(valid_A, trainmedians=train_medians_A)
test_A, _  = median_imputation(test_A,  trainmedians=train_medians_A)

train_A, norm_stats_A = normalize(train_A)
valid_A, _ = normalize(valid_A, trainstats=norm_stats_A)
test_A, _  = normalize(test_A,  trainstats=norm_stats_A)

actual_cols_A = [c for c in train_A.columns
                 if c not in ('row_id', 'mode', 'part', 'entity_id', 'timestamp', 'y')]
for col in actual_cols_A:
    if col not in valid_A.columns: valid_A[col] = 0
    if col not in test_A.columns:  test_A[col] = 0

X_train_A = createX_bias(train_A, actual_cols_A)
y_train_A = createy(train_A)
X_valid_A = createX_bias(valid_A, actual_cols_A)
y_valid_A = createy(valid_A)
X_test_A  = createX_bias(test_A,  actual_cols_A)

best_lambda_A = 95.40954763499944
w_A = ridge_closed_form(X_train_A, y_train_A, lamda=best_lambda_A)
print(f"Part A - Train MAE: {mae(y_train_A, predict(w_A, X_train_A)):.4f}")
print(f"Part A - Valid MAE: {mae(y_valid_A, predict(w_A, X_valid_A)):.4f}")

# PART B — Polynomial Ridge Regression

train_B = train_df[train_df['part'] == 'B'].copy()
valid_B = valid_df[valid_df['part'] == 'B'].copy()
test_B  = test_df[test_df['part'] == 'B'].copy()

unique_B = extract_unique(train_B, categorical_cols_AB)
train_B = one_hot(train_B, categorical_cols_AB, unique_B)
valid_B = one_hot(valid_B, categorical_cols_AB, unique_B)
test_B  = one_hot(test_B,  categorical_cols_AB, unique_B)

train_B, train_medians_B = median_imputation(train_B)
valid_B, _ = median_imputation(valid_B, trainmedians=train_medians_B)
test_B, _  = median_imputation(test_B,  trainmedians=train_medians_B)

train_B, norm_stats_B = normalize(train_B)
valid_B, _ = normalize(valid_B, trainstats=norm_stats_B)
test_B, _  = normalize(test_B,  trainstats=norm_stats_B)

actual_cols_B = [c for c in train_B.columns
                 if c not in ('row_id', 'mode', 'part', 'entity_id', 'timestamp', 'y')]

valid_B = valid_B.reindex(columns=train_B.columns, fill_value=0)
test_B  = test_B.reindex(columns=train_B.columns, fill_value=0)

X_train_B = createX_no_bias(train_B, actual_cols_B)
y_train_B = createy(train_B)
X_valid_B = createX_no_bias(valid_B, actual_cols_B)
y_valid_B = createy(valid_B)
X_test_B  = createX_no_bias(test_B,  actual_cols_B)

X_train_B_poly = add_bias(build_custom_poly_features(X_train_B))
X_valid_B_poly = add_bias(build_custom_poly_features(X_valid_B))
X_test_B_poly  = add_bias(build_custom_poly_features(X_test_B))

best_lambda_B = 1778.2794
w_B = ridge_closed_form(X_train_B_poly, y_train_B, lamda=best_lambda_B)
print(f"Part B - Train MAE: {mae(y_train_B, predict(w_B, X_train_B_poly)):.4f}")
print(f"Part B - Valid MAE: {mae(y_valid_B, predict(w_B, X_valid_B_poly)):.4f}")

# PART C — Ridge Regression (normal normalization)

train_C = train_df[train_df['part'] == 'C'].copy()
valid_C = valid_df[valid_df['part'] == 'C'].copy()
test_C  = test_df[test_df['part'] == 'C'].copy()

train_C = train_C.sort_values(by=['entity_id', 'timestamp']).reset_index(drop=True)
valid_C = valid_C.sort_values(by=['entity_id', 'timestamp']).reset_index(drop=True)
test_C  = test_C.sort_values(by=['entity_id', 'timestamp']).reset_index(drop=True)

unique_C = extract_unique(train_C, categorical_cols_CD)
train_C = one_hot(train_C, categorical_cols_CD, unique_C)
valid_C = one_hot(valid_C, categorical_cols_CD, unique_C)
test_C  = one_hot(test_C,  categorical_cols_CD, unique_C)

train_C, train_medians_C = median_imputation(train_C)
valid_C, _ = median_imputation(valid_C, trainmedians=train_medians_C)
test_C, _  = median_imputation(test_C,  trainmedians=train_medians_C)

train_C, norm_stats_C = normalize(train_C)
valid_C, _ = normalize(valid_C, trainstats=norm_stats_C)
test_C, _  = normalize(test_C,  trainstats=norm_stats_C)

actual_cols_C = [c for c in train_C.columns
                 if c not in ('row_id', 'mode', 'part', 'entity_id', 'timestamp', 'y')]
for col in actual_cols_C:
    if col not in valid_C.columns: valid_C[col] = 0
    if col not in test_C.columns:  test_C[col] = 0

X_train_C = createX_bias(train_C, actual_cols_C)
y_train_C = createy(train_C)
X_valid_C = createX_bias(valid_C, actual_cols_C)
y_valid_C = createy(valid_C)
X_test_C  = createX_bias(test_C,  actual_cols_C)

best_lambda_C = 255.95479226995332
w_C = ridge_closed_form(X_train_C, y_train_C, lamda=best_lambda_C)
print(f"Part C - Train MAE: {mae(y_train_C, predict(w_C, X_train_C)):.4f}")
print(f"Part C - Valid MAE: {mae(y_valid_C, predict(w_C, X_valid_C)):.4f}")

# PART D — Ridge Regression (same as Part C) + Y-Lag Features + Robust Normalization

train_D = train_df[train_df['part'] == 'D'].copy()
valid_D = valid_df[valid_df['part'] == 'D'].copy()
test_D  = test_df[test_df['part'] == 'D'].copy()

train_D = train_D.sort_values(by=['entity_id', 'timestamp']).reset_index(drop=True)
valid_D = valid_D.sort_values(by=['entity_id', 'timestamp']).reset_index(drop=True)
test_D  = test_D.sort_values(by=['entity_id', 'timestamp']).reset_index(drop=True)

# Add Y-Lag Features 
def add_ylag_features(df):
    df = df.copy()
    eps = 1e-6
    lag1 = df['y_lag1'].fillna(df['y_lag1'].median())
    lag7 = df['y_lag7'].fillna(df['y_lag7'].median())

    df['ylag_product']    = lag1 * lag7
    df['ylag1_sq']        = lag1 ** 2
    df['ylag7_sq']        = lag7 ** 2
    df['log_ylag1']       = np.log(np.abs(lag1) + eps)
    df['log_ylag7']       = np.log(np.abs(lag7) + eps)
    df['log_ylag_product']= np.log(np.abs(lag1) + eps) * np.log(np.abs(lag7) + eps)
    df['log_ylag_ratio']  = np.log((np.abs(lag1) + eps) / (np.abs(lag7) + eps))
    df['ylag_diff']       = lag1 - lag7
    df['ylag_ratio']      = lag1 / (np.abs(lag7) + eps)
    df['ylag_abs_diff']   = np.abs(lag1 - lag7)
    df['ylag_mean']       = (lag1 + lag7) / 2.0
    return df

train_D = add_ylag_features(train_D)
valid_D = add_ylag_features(valid_D)
test_D  = add_ylag_features(test_D)

# Entity Mean
entity_means  = train_D.groupby('entity_id')['y'].mean().to_dict()
global_mean_D = train_D['y'].mean()
train_D['entity_mean_y'] = train_D['entity_id'].map(entity_means).fillna(global_mean_D)
valid_D['entity_mean_y'] = valid_D['entity_id'].map(entity_means).fillna(global_mean_D)
test_D['entity_mean_y']  = test_D['entity_id'].map(entity_means).fillna(global_mean_D)

# One Hot & Imputation 
categorical_cols_D = ['dow', 'month_phase', 'dept', 'sensor_class', 'building_type',
                      'entity_id', 'holiday_flag']
unique_D = extract_unique(train_D, categorical_cols_D)
train_D = one_hot(train_D, categorical_cols_D, unique_D)
valid_D = one_hot(valid_D, categorical_cols_D, unique_D)
test_D  = one_hot(test_D,  categorical_cols_D, unique_D)

train_D, train_medians_D = median_imputation(train_D)
valid_D, _               = median_imputation(valid_D, train_medians_D)
test_D,  _               = median_imputation(test_D,  train_medians_D)

# Robust Normalize ( covers x-cols + y_lag1/y_lag7/timestamp)
train_D, norm_stats_D = robust_normalize(train_D)
valid_D, _            = robust_normalize(valid_D, stats=norm_stats_D)
test_D,  _            = robust_normalize(test_D,  stats=norm_stats_D)

# Clip & Normalize new Y-Lag engineered columns 
ylag_cols = ['ylag_product', 'ylag1_sq', 'ylag7_sq', 'log_ylag1', 'log_ylag7',
             'log_ylag_product', 'log_ylag_ratio', 'ylag_diff', 'ylag_ratio',
             'ylag_abs_diff', 'ylag_mean']

ylag_clip_stats_D = {}
ylag_norm_stats_D = {}
for col in ylag_cols:
    mu, sigma = train_D[col].mean(), train_D[col].std()
    ylag_clip_stats_D[col] = (mu - 3*sigma, mu + 3*sigma)
    ylag_norm_stats_D[col] = (mu, sigma)

    for df in [train_D, valid_D, test_D]:
        df[col] = df[col].clip(lower=ylag_clip_stats_D[col][0], upper=ylag_clip_stats_D[col][1])
        if sigma < 1e-8 or pd.isna(mu) or pd.isna(sigma):
            df[col] = 0.0
        else:
            df[col] = (df[col] - mu) / sigma
            df[col] = df[col].fillna(0.0)

#  Prepare Matrices 
actual_cols_D = [c for c in train_D.columns
                 if c not in ('row_id', 'mode', 'part', 'entity_id', 'timestamp', 'y')]
for col in actual_cols_D:
    if col not in valid_D.columns: valid_D[col] = 0
    if col not in test_D.columns:  test_D[col] = 0

X_train_D = createX_bias(train_D, actual_cols_D)
y_train_D = createy(train_D)
X_valid_D = createX_bias(valid_D, actual_cols_D)
y_valid_D = createy(valid_D)
X_test_D  = createX_bias(test_D,  actual_cols_D)

# Hardcoded lambda for Part D 
best_lambda_D = 1.019641
print(f"Part D - Hardcoded lambda: {best_lambda_D:.6f}")

w_D = ridge_closed_form(X_train_D, y_train_D, lamda=best_lambda_D)
print(f"Part D - Train MAE: {mae(y_train_D, predict(w_D, X_train_D)):.4f}")
print(f"Part D - Valid MAE: {mae(y_valid_D, predict(w_D, X_valid_D)):.4f}")


# FINAL SUBMISSION GENERATION (Retrain on Train + Valid)

print("\n=== Starting Final Submission Generation (Train + Valid) ===")

full_train_df = pd.concat([train_df, valid_df], ignore_index=True)
full_train_A = full_train_df[full_train_df['part'] == 'A'].copy()
full_train_B = full_train_df[full_train_df['part'] == 'B'].copy()
full_train_C = full_train_df[full_train_df['part'] == 'C'].copy()
full_train_D = full_train_df[full_train_df['part'] == 'D'].copy()

test_A_raw = test_df[test_df['part'] == 'A'].copy()
test_B_raw = test_df[test_df['part'] == 'B'].copy()
test_C_raw = test_df[test_df['part'] == 'C'].copy()
test_D_raw = test_df[test_df['part'] == 'D'].copy()

final_predictions = np.zeros(len(test_df))

#  Part A Final 
unique_A_final = extract_unique(full_train_A, categorical_cols_AB)
full_train_A = one_hot(full_train_A, categorical_cols_AB, unique_A_final)
test_A_final = one_hot(test_A_raw, categorical_cols_AB, unique_A_final)
full_train_A, train_medians_A_final = median_imputation(full_train_A)
test_A_final, _ = median_imputation(test_A_final, trainmedians=train_medians_A_final)
full_train_A, norm_stats_A_final = normalize(full_train_A)
test_A_final, _ = normalize(test_A_final, trainstats=norm_stats_A_final)
actual_cols_A_final = [c for c in full_train_A.columns
                       if c not in ('row_id', 'mode', 'part', 'entity_id', 'timestamp', 'y')]
for col in actual_cols_A_final:
    if col not in test_A_final.columns: test_A_final[col] = 0
X_train_A_final = createX_bias(full_train_A, actual_cols_A_final)
y_train_A_final = createy(full_train_A)
X_test_A_final  = createX_bias(test_A_final, actual_cols_A_final)
w_A_final = ridge_closed_form(X_train_A_final, y_train_A_final, lamda=best_lambda_A)
final_predictions[test_A_raw.index] = predict(w_A_final, X_test_A_final)
print(f"Part A Final Predictions Generated ({len(test_A_raw)} rows)")

#  Part B Final 
unique_B_final = extract_unique(full_train_B, categorical_cols_AB)
full_train_B = one_hot(full_train_B, categorical_cols_AB, unique_B_final)
test_B_final = one_hot(test_B_raw, categorical_cols_AB, unique_B_final)
full_train_B, train_medians_B_final = median_imputation(full_train_B)
test_B_final, _ = median_imputation(test_B_final, trainmedians=train_medians_B_final)
full_train_B, norm_stats_B_final = normalize(full_train_B)
test_B_final, _ = normalize(test_B_final, trainstats=norm_stats_B_final)
actual_cols_B_final = [c for c in full_train_B.columns
                       if c not in ('row_id', 'mode', 'part', 'entity_id', 'timestamp', 'y')]
test_B_final = test_B_final.reindex(columns=full_train_B.columns, fill_value=0)
X_train_B_final = createX_no_bias(full_train_B, actual_cols_B_final)
y_train_B_final = createy(full_train_B)
X_test_B_final  = createX_no_bias(test_B_final, actual_cols_B_final)
X_train_B_final_poly = add_bias(build_custom_poly_features(X_train_B_final))
X_test_B_final_poly  = add_bias(build_custom_poly_features(X_test_B_final))
w_B_final = ridge_closed_form(X_train_B_final_poly, y_train_B_final, lamda=best_lambda_B)
final_predictions[test_B_raw.index] = predict(w_B_final, X_test_B_final_poly)
print(f"Part B Final Predictions Generated ({len(test_B_raw)} rows)")

#  Part C Final 
full_train_C = full_train_C.sort_values(by=['entity_id', 'timestamp']).reset_index(drop=True)
test_C_sorted_with_idx = test_C_raw.sort_values(by=['entity_id', 'timestamp']).reset_index(drop=False)
unique_C_final = extract_unique(full_train_C, categorical_cols_CD)
full_train_C = one_hot(full_train_C, categorical_cols_CD, unique_C_final)
test_C_final_align = one_hot(test_C_sorted_with_idx, categorical_cols_CD, unique_C_final)
full_train_C, train_medians_C_final = median_imputation(full_train_C)
test_C_final_align, _ = median_imputation(test_C_final_align, trainmedians=train_medians_C_final)
full_train_C, norm_stats_C_final = normalize(full_train_C)
test_C_final_align, _ = normalize(test_C_final_align, trainstats=norm_stats_C_final)
actual_cols_C_final = [c for c in full_train_C.columns
                       if c not in ('row_id', 'mode', 'part', 'entity_id', 'timestamp', 'y')]
for col in actual_cols_C_final:
    if col not in test_C_final_align.columns: test_C_final_align[col] = 0
X_train_C_final = createX_bias(full_train_C, actual_cols_C_final)
y_train_C_final = createy(full_train_C)
X_test_C_final  = createX_bias(test_C_final_align, actual_cols_C_final)
w_C_final = ridge_closed_form(X_train_C_final, y_train_C_final, lamda=best_lambda_C)
final_predictions[test_C_sorted_with_idx['index']] = X_test_C_final @ w_C_final
print(f"Part C Final Predictions Generated ({len(test_C_raw)} rows)")

#  Part D Final 
full_train_D = full_train_D.sort_values(by=['entity_id', 'timestamp']).reset_index(drop=True)
test_D_sorted_with_idx = test_D_raw.sort_values(by=['entity_id', 'timestamp']).reset_index(drop=False)

# Add ylag features
full_train_D = add_ylag_features(full_train_D)
test_D_sorted_with_idx = add_ylag_features(test_D_sorted_with_idx)

# Entity mean
entity_means_final  = full_train_D.groupby('entity_id')['y'].mean().to_dict()
global_mean_D_final = full_train_D['y'].mean()
full_train_D['entity_mean_y'] = full_train_D['entity_id'].map(entity_means_final).fillna(global_mean_D_final)
test_D_sorted_with_idx['entity_mean_y'] = test_D_sorted_with_idx['entity_id'].map(entity_means_final).fillna(global_mean_D_final)

# One hot & Imputation
unique_D_final = extract_unique(full_train_D, categorical_cols_D)
full_train_D = one_hot(full_train_D, categorical_cols_D, unique_D_final)
test_D_final_align = one_hot(test_D_sorted_with_idx, categorical_cols_D, unique_D_final)

full_train_D, train_medians_D_final = median_imputation(full_train_D)
test_D_final_align, _ = median_imputation(test_D_final_align, train_medians_D_final)

# Robust Normalize
full_train_D, norm_stats_D_final = robust_normalize(full_train_D)
test_D_final_align, _ = robust_normalize(test_D_final_align, stats=norm_stats_D_final)

# Clip & Normalize Y-Lag engineered columns (reusing full_train stats)
for col in ylag_cols:
    mu, sigma = full_train_D[col].mean(), full_train_D[col].std()
    lo, hi = mu - 3*sigma, mu + 3*sigma
    full_train_D[col] = full_train_D[col].clip(lower=lo, upper=hi)
    test_D_final_align[col] = test_D_final_align[col].clip(lower=lo, upper=hi)

    if sigma < 1e-8 or pd.isna(mu) or pd.isna(sigma):
        full_train_D[col] = 0.0
        test_D_final_align[col] = 0.0
    else:
        full_train_D[col] = (full_train_D[col] - mu) / sigma
        test_D_final_align[col] = (test_D_final_align[col] - mu) / sigma
        full_train_D[col] = full_train_D[col].fillna(0.0)
        test_D_final_align[col] = test_D_final_align[col].fillna(0.0)

# Prepare final matrices
actual_cols_D_final = [c for c in full_train_D.columns
                       if c not in ('row_id', 'mode', 'part', 'entity_id', 'timestamp', 'y')]
for col in actual_cols_D_final:
    if col not in test_D_final_align.columns: test_D_final_align[col] = 0

X_train_D_final = createX_bias(full_train_D, actual_cols_D_final)
y_train_D_final = createy(full_train_D)
X_test_D_final  = createX_bias(test_D_final_align, actual_cols_D_final)

w_D_final = ridge_closed_form(X_train_D_final, y_train_D_final, lamda=best_lambda_D)
final_predictions[test_D_sorted_with_idx['index']] = predict(w_D_final, X_test_D_final)
print(f"Part D Final Predictions Generated ({len(test_D_raw)} rows)")

#  Create Submission File 
submission_df = pd.DataFrame({
    'row_id': test_df['row_id'],
    'y'     : final_predictions
})
submission_df.to_csv('submission.csv', index=False)
print("Submission file 'submission.csv' created successfully.")
print(submission_df.head())
